In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. TUNING PRESETS & AIRCRAFT PITCH MODEL
# ==========================================
TUNING_PRESETS = {
    "Balanced Tune":          {"kp": 2.40, "ki": 0.90, "kd": 1.15, "note": "Well-balanced gains with fast response, minimal overshoot, and zero steady-state error."},
    "Low Kp (Sluggish)":      {"kp": 0.65, "ki": 0.40, "kd": 0.45, "note": "Weak proportional gain makes pitch response slow and under-powered."},
    "High Kp (Oscillatory)":  {"kp": 7.50, "ki": 1.00, "kd": 0.60, "note": "Excessive proportional gain over-drives elevator deflections, causing strong overshoot and ringing."},
    "Low Ki (Steady Offset)": {"kp": 2.20, "ki": 0.02, "kd": 1.00, "note": "Near-zero integral gain leaves a persistent steady-state pitch error."},
    "High Ki (Windup/Unstable)":{"kp":2.60,"ki": 5.50, "kd": 0.80, "note": "Excessive integral gain accumulates error (windup) and triggers slow/growing oscillations."},
    "Low Kd (Underdamped)":   {"kp": 3.20, "ki": 0.90, "kd": 0.05, "note": "Insufficient derivative damping lets pitch bounce repeatedly before settling."},
}

def simulate_pitch_pid(target_deg, kp, ki, kd, duration_s=10.0, dt=0.01):
    n = int(duration_s / dt) + 1
    t = np.linspace(0.0, duration_s, n)

    # Pitch dynamics proxy: d(theta)/dt = q,  dq/dt = k_elev * elevator - k_damp * q - k_stab * theta
    theta = np.zeros(n)
    q = np.zeros(n)
    elev = np.zeros(n)

    integral_err = 0.0
    prev_err = target_deg - theta[0]

    for i in range(1, n):
        err = target_deg - theta[i - 1]
        integral_err = np.clip(integral_err + err * dt, -40.0, 40.0)
        deriv_err = (err - prev_err) / dt
        prev_err = err

        u_cmd = kp * err + ki * integral_err + kd * deriv_err
        # Elevator deflection saturation [-25 deg, +25 deg]
        elev[i] = np.clip(0.35 * u_cmd, -25.0, 25.0)

        # Aircraft short-period-like pitch response
        q_dot = 2.6 * elev[i] - 1.35 * q[i - 1] - 1.10 * theta[i - 1]
        q[i] = np.clip(q[i - 1] + q_dot * dt, -120.0, 120.0)
        theta[i] = np.clip(theta[i - 1] + q[i] * dt, -45.0, 60.0)

    final_theta = float(np.mean(theta[-int(1.0 / dt):]))
    ss_err = float(target_deg - final_theta)
    peak_theta = float(np.max(theta)) if target_deg >= 0 else float(np.min(theta))

    if abs(target_deg) > 1e-6:
        overshoot_pct = max(0.0, ((peak_theta - target_deg) / abs(target_deg)) * 100.0) if target_deg > 0 else max(0.0, ((target_deg - peak_theta) / abs(target_deg)) * 100.0)
    else:
        overshoot_pct = abs(peak_theta) * 10.0

    # Oscillation indicator (zero-crossings of error in second half)
    err_half = target_deg - theta[n // 2:]
    zero_cross = int(np.sum(np.diff(np.signbit(err_half)).astype(int)))

    return pd.DataFrame({
        "time_s": t,
        "target_deg": np.full(n, target_deg),
        "pitch_deg": theta,
        "elevator_deg": elev,
    }), {
        "final_theta": final_theta,
        "ss_err": ss_err,
        "peak_theta": peak_theta,
        "overshoot_pct": overshoot_pct,
        "zero_cross": zero_cross,
    }

def build_pid_why_explanation(preset_name, kp, ki, kd, stats, target_deg):
    reasons = []
    if kp < 1.1:
        reasons.append(f"low Kp ({kp:.2f}) gives a gentle control push, causing a slow rise toward {target_deg:.1f}°")
    elif kp > 5.0:
        reasons.append(f"high Kp ({kp:.2f}) reacts aggressively to error, driving strong overshoot ({stats['overshoot_pct']:.1f}%) and oscillations")

    if ki < 0.15:
        reasons.append(f"very low Ki ({ki:.2f}) leaves a steady-state pitch offset ({stats['ss_err']:+.2f}°)")
    elif ki > 3.5:
        reasons.append(f"high Ki ({ki:.2f}) over-accumulates integral error, creating heavy low-frequency swing")

    if kd < 0.20:
        reasons.append(f"low Kd ({kd:.2f}) provides minimal rate damping, allowing {stats['zero_cross']} error crossings in the second half")

    if not reasons:
        reasons.append(f"Kp ({kp:.2f}), Ki ({ki:.2f}), and Kd ({kd:.2f}) work together cleanly with {stats['overshoot_pct']:.1f}% overshoot and {stats['ss_err']:+.2f}° steady-state error")

    return f"<b>Why This Result Happened ({preset_name}):</b> The pitch response behaves this way because " + "; and ".join(reasons) + "."

# ==========================================
# 2. 2D AIRCRAFT PITCH VIEW + TIME RESPONSE
# ==========================================
def draw_pitch_pid(preset_name, target_deg, kp, ki, kd):
    df, stats = simulate_pitch_pid(target_deg, kp, ki, kd)
    why_text = build_pid_why_explanation(preset_name, kp, ki, kd, stats, target_deg)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_2d, ax_ts) = plt.subplots(1, 2, figsize=(13.6, 4.6), gridspec_kw={"width_ratios": [1.0, 1.45]})

    # --- LEFT: 2D Aircraft Pitch Attitude Schematic ---
    actual_rad = np.deg2rad(stats["final_theta"])
    target_rad = np.deg2rad(target_deg)

    # Horizontal datum
    ax_2d.plot([-1.3, 1.3], [0.0, 0.0], color="#94a3b8", ls="--", lw=1.2, label="Horizon (0°)")

    # Target pitch line
    ax_2d.plot([-1.1 * np.cos(target_rad), 1.1 * np.cos(target_rad)],
               [-1.1 * np.sin(target_rad), 1.1 * np.sin(target_rad)],
               color="#16a34a", ls=":", lw=1.8, label=f"Target Pitch ({target_deg:+.1f}°)")

    # Aircraft fuselage & wing line at final pitch
    nose_x, nose_y = 1.05 * np.cos(actual_rad), 1.05 * np.sin(actual_rad)
    tail_x, tail_y = -1.05 * np.cos(actual_rad), -1.05 * np.sin(actual_rad)
    ax_2d.plot([tail_x, nose_x], [tail_y, nose_y], color="#2563eb", lw=4.0, solid_capstyle="round", label=f"Actual Pitch ({stats['final_theta']:+.1f}°)")
    ax_2d.scatter([nose_x], [nose_y], color="#dc2626", s=65, zorder=6, label="Nose")

    # Wing bar perpendicular to fuselage
    wing_span = 0.42
    wx1, wy1 = -wing_span * np.sin(actual_rad), wing_span * np.cos(actual_rad)
    wx2, wy2 = wing_span * np.sin(actual_rad), -wing_span * np.cos(actual_rad)
    ax_2d.plot([wx1, wx2], [wy1, wy2], color="#0f172a", lw=3.0)

    ax_2d.set_title("2D Aircraft Pitch Attitude (End State)", fontsize=11, fontweight="bold", pad=10)
    ax_2d.set_xlim(-1.35, 1.35)
    ax_2d.set_ylim(-1.15, 1.15)
    ax_2d.set_aspect("equal", adjustable="box")
    ax_2d.axis("off")
    ax_2d.legend(loc="lower center", ncol=2, fontsize=8, frameon=True, facecolor="white")

    # --- RIGHT: Pitch Angle vs. Time ---
    ax_ts.plot(df["time_s"], df["target_deg"], color="#16a34a", ls="--", lw=1.8, label="Target Pitch [deg]")
    ax_ts.plot(df["time_s"], df["pitch_deg"], color="#2563eb", lw=2.2, label="Actual Pitch [deg]")

    ax_ts.set_title(f"Aircraft Pitch Response — Kp={kp:.2f}, Ki={ki:.2f}, Kd={kd:.2f}", fontsize=11, fontweight="bold", pad=10)
    ax_ts.set_xlabel("Time [s]", fontsize=9.5)
    ax_ts.set_ylabel("Pitch Angle [deg]", fontsize=9.5)
    ax_ts.legend(loc="lower right", fontsize=8.5, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:13px 17px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="font-size:13px;">
            <b>Preset:</b> <span style="color:#7dd3fc;font-weight:700;">{preset_name}</span> &nbsp;|&nbsp;
            <b>Final Pitch:</b> {stats['final_theta']:+.2f}° &nbsp;|&nbsp;
            <b>Overshoot:</b> {stats['overshoot_pct']:.1f}% &nbsp;|&nbsp;
            <b>Steady-State Error:</b> {stats['ss_err']:+.2f}°
        </div>
        <div style="font-size:12.5px;color:#e2e8f0;margin-top:6px;line-height:1.55;">
            {why_text}
        </div>
        <div style="font-size:12px;color:#94a3b8;margin-top:4px;line-height:1.45;">
            {TUNING_PRESETS[preset_name]['note']}
        </div>
    </div>
    """))

# ==========================================
# 3. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "145px"}
ctrl_layout = widgets.Layout(width="340px")

w_preset = widgets.Dropdown(options=list(TUNING_PRESETS.keys()), value="Balanced Tune", description="Tuning Preset:", style=lbl_style, layout=ctrl_layout)
w_target = widgets.FloatSlider(value=12.0, min=-15.0, max=25.0, step=1.0, description="Target Pitch [deg]:", readout_format="+.1f", style=lbl_style, layout=ctrl_layout)
w_kp     = widgets.FloatSlider(value=2.40, min=0.00,  max=10.0, step=0.10, description="Kp (Proportional):", readout_format=".2f", style=lbl_style, layout=ctrl_layout)
w_ki     = widgets.FloatSlider(value=0.90, min=0.00,  max=6.00, step=0.05, description="Ki (Integral):", readout_format=".2f", style=lbl_style, layout=ctrl_layout)
w_kd     = widgets.FloatSlider(value=1.15, min=0.00,  max=4.00, step=0.05, description="Kd (Derivative):", readout_format=".2f", style=lbl_style, layout=ctrl_layout)

btn_sim = widgets.Button(description="Run Pitch Simulation", button_style="primary", icon="plane", layout=widgets.Layout(width="220px", height="36px"))
out_pid = widgets.Output(layout=widgets.Layout(width="100%"))

syncing = {"flag": False}

def apply_preset(change):
    if syncing["flag"]:
        return
    syncing["flag"] = True
    p = TUNING_PRESETS[change["new"]]
    w_kp.value = p["kp"]
    w_ki.value = p["ki"]
    w_kd.value = p["kd"]
    syncing["flag"] = False
    refresh_pid()

def refresh_pid(*args):
    if syncing["flag"]:
        return
    with out_pid:
        clear_output(wait=True)
        draw_pitch_pid(w_preset.value, w_target.value, w_kp.value, w_ki.value, w_kd.value)

w_preset.observe(apply_preset, names="value")
btn_sim.on_click(refresh_pid)
for c in [w_target, w_kp, w_ki, w_kd]:
    c.observe(refresh_pid, names="value")

header_pid = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Aircraft Pitch PID Controller &amp; Bad-Tuning Explorer</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Simulate pitch attitude control and observe how wrong Kp, Ki, or Kd tuning affects response
        </div>
    </div>
""")

controls_pid = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_preset, w_target, w_kp], layout=widgets.Layout(margin="0 28px 0 0", gap="8px")),
        widgets.VBox([w_ki, w_kd, btn_sim], layout=widgets.Layout(gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_pid, controls_pid, out_pid)
refresh_pid()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))